In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\srija\OneDrive\Desktop\machine learning\DEEP LEARNING\datasets\DateFruit_Dataset.csv")

In [ ]:
df.isnull().sum()
df.info()
df.head(5)

In [ ]:
X = df.drop(columns=["Class"], axis=1)
y = df["Class"] 

In [ ]:
df["Class"].unique()

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scale = scaler.fit_transform(X_train)
X_test_scale = scaler.transform(X_test)

### Deep Learning

In [ ]:
# ANN
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
X_train_tensor = torch.tensor(X_train_scale, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scale, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
# target values (y) must be in long format because we do cross-entropy-loss for loss function, and it expects long format.

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [ ]:
# Build our model

class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model  = nn.Sequential(

            nn.Linear(X.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 7)
        )
    
    def forward(self, x):
        return self.model(x)

In [ ]:
model = ANN()

#loss and optim
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [ ]:
# Training the NN

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss = 0.0
    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)
        loss.backward()
        optimizer.step() # parameters update

        running_loss += loss

    train_loss = running_loss / len(train_loader)
    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

In [ ]:
# Evaluation, and fixing steps
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1) # <- (max_value, max_value_index), _, means dont save the value

        correct += (predicted == yb).sum().item()
        total += yb.size(0) # return actual samples in each batch.

print("total values: ", total)
print("correct values: ", correct)
print("Accuracy: ", correct/total * 100)